# Notebook 06: Faster R-CNN

**Module:** 08 Object Detection  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand Region Proposal Network (RPN)
2. Know anchor boxes at each feature map location
3. Run torchvision Faster R-CNN on synthetic data
4. Apply to vehicle/building detection

## Faster R-CNN (Ren et al., 2015)

**RPN:** Small network on feature map predicts objectness + box offsets for **anchors** at each spatial location.

**Two heads after RPN:**
1. RPN → proposals
2. RoI head → class + refined box

**Anchors:** Multiple scales/aspect ratios per pixel (e.g. 9 anchors at each of 50×38 locations).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
try:
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    def build_faster_rcnn(num_classes=2):
        model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        return model

    model = build_faster_rcnn(num_classes=2)  # background + object
    model.eval()
    images = [torch.rand(3, 256, 256)]
    with torch.no_grad():
        out = model(images)
    print('Faster R-CNN output keys:', out[0].keys())
    print('Num detections:', len(out[0]['boxes']))
except ImportError as e:
    print('torchvision detection requires recent torchvision:', e)

## GeoSpatial Use Case

**Building detection:** Faster R-CNN on satellite tiles — good when objects are sparse and precise boxes needed.

**vs Segmentation:** Detection counts instances; segmentation delineates boundaries.

---

## Summary

Faster R-CNN = RPN proposals + RoI head — two-stage SOTA baseline for years.

**Next:** [07_SSD.ipynb](07_SSD.ipynb)